# Recap

1. j'ai utilisé les llm pour faire l'extraction de données (feedable dans ai-rchivist)
2. j'ai fine tuné xlm-roberta sur le corpus de l'univ de Tours pour être capable de faire de la NER
3. j'ai demandé aux gens de labelliser le corpus
4. j'ai fine tuné plusieurs modèles pour faire de la NER
    * xaviergillard/xlm-roberta-vieille-france-v2   (fine tuning)
    * xaviergillard/xlm-roberta-vieille-france-v2.1 (idem, plus d'epochs)
    * xaviergillard/brabrant-xvii-ner (xlmr entrainé directement sur les données)
    * xaviergillard/brabrant-scratch-ner (bert custom entrainé sur les données) 

In [1]:
import json
import typing
import pandas as pd
import numpy  as np
import torch

from pathlib         import Path
from pydantic        import BaseModel
from transformers    import AutoTokenizer, AutoModelForTokenClassification
from datasets        import Dataset
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

/home/xgillard/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: on prépare le gold standard pour évaluer

In [2]:
class Annotation(BaseModel):
    start:  int
    end:    int
    labels: list[str]
    text:   str

    @property
    def label(self):
        return self.labels[0]
        
    def __lt__(self, other) -> bool:
        return ( (self.start < other.start)
            or   ((self.start == other.start) and (self.end < other.end)) )

In [3]:
def simplify(item):
    return {
        "id"      : item["data"]["id"],
        "text"    : item["data"]["text"],
        "annotations": sorted([ Annotation.model_validate(r["value"]) for a in item["annotations"] for r in a["result"]])
    }

def align(text: str, annotations: list[Annotation]) -> tuple[list[str], list[str]]:
    classes= {'PERSON': 'PERS', 'LOCATION': 'LOC', 'DATE': 'DATE', 'O': 'O'}
    tokens = text.split()
    labels = [ 'O' for _ in tokens ]

    annots = iter(annotations)
    # on assigne grossierement les labels
    token = 0
    try:
        annot = next(annots)
        for i,c in enumerate(text):
            if i > 0 and c.isspace() and not text[i-1].isspace():
                token += 1
            if i >= annot.start and i < annot.end:
                labels[token] = classes[annot.label]
            if i >= annot.end:
                annot = next(annots)
    except StopIteration:
        pass
    # on s'assure de respecter le protocole IOB
    for i,l in enumerate(labels):
        if l == 'O':
            continue
        if i == 0:
            labels[i] = f'B-{l}'
        prev = labels[i-1]
        if prev == f'B-{l}' or prev == f'I-{l}':
            labels[i] = f'I-{l}'
        else:
            labels[i] = f'I-{l}'
    # ok on a fini
    return tokens, labels

def process_item(item):
    item = simplify(item)
    x,y  = align(item['text'], item['annotations'])
    return {'id': item["id"], 'tokens': x, 'labels': y}

gold = [process_item(item) for item in json.loads(Path("GOLD_STD_LABELS.json").read_text(encoding="utf8")) ]
gold = pd.DataFrame(gold, columns=["id", "tokens", "labels"]).set_index(["id"])
gold[:5]

,tokens,labels
id,,
279,"[pardon, moyennisse, amende, kaerle, etc., all...","[O, O, O, I-PERS, O, O, O, O, O, O, O, O, O, O..."
313,"[kaerle, etc., doen, te, weeten, allen, tegenw...","[I-PERS, O, O, O, O, O, O, O, O, O, O, O, O, O..."
416,"[kaerle, etc., doen, te, wetene, allen, iegewo...","[I-PERS, O, O, O, O, O, O, O, O, O, O, O, O, O..."
390,"[philips, etc, doen, te, wetene, dat, wij, heb...","[I-PERS, O, O, O, O, O, O, O, O, O, O, O, I-PE..."
72,"[remissie, charles, etc, savoir, faisons, a, t...","[O, I-PERS, O, O, O, O, O, O, O, O, O, O, O, O..."


## Step 2: 

In [4]:
device     = "cuda" if torch.cuda.is_available() else "cpu"

checkpoint = "xaviergillard/xlm-roberta-large-vieille-france"
tokenizer  = AutoTokenizer.from_pretrained(checkpoint)
model      = AutoModelForTokenClassification.from_pretrained(checkpoint).eval().to(device)

In [5]:
def make_dataset(tokenizer, gold):
    """
    Crée le dataset correspondant au corpus complet tokenizé avec les labels qui sont correctement
    alignés sur les tokens. Ca renvoie un dataset avec les colonnes suivantes: 
    - input_ids
    - attention_mask
    - labels
    - tokens
    - tags
    """
    # on commence par tout tokenizer en batch
    max_seq_len= tokenizer.model_max_length
    tokenized  = tokenizer(
        gold['tokens'].tolist(), 
        is_split_into_words=True,
        padding=True, 
        truncation=True,
        max_length=max_seq_len,
        return_overflowing_tokens=True,
    )
    # on aligne les labels avec les tokens dans les différentes séquences qui découpent le texte
    tokids = []
    labels = []
    amasks = []
    tokens = []
    classes= []
    for i, token_ids in enumerate(tokenized['input_ids']):
        sample_id  = tokenized['overflow_to_sample_mapping'][i]
        encoding   = tokenized.encodings[i]
        word_ids   = encoding.word_ids
        decoded    = [ tokenizer.convert_ids_to_tokens(tid) for tid in token_ids ]
        labelclass = [ gold.iloc[sample_id]['labels'][w] if w is not None else 'IGN' for w in word_ids ]
        # on fait une passe pour corriger les B- labels qui appartiennent au meme mot.
        for k,wid in enumerate(word_ids):
            if k == 0:
                continue
            if wid == word_ids[k-1] and labelclass[k].startswith('B-'):
                labelclass[k] = 'I-' + labelclass[k][2:]
        # on construit la liste d'id pour les labels
        labelids   = [ model.config.label2id[l] for l in labelclass ]
        amasks .append(tokenized['attention_mask'][i])
        tokens .append(decoded)
        tokids .append(token_ids)
        classes.append(labelclass)
        labels .append(labelids)
    # on convertit le tout en un dataset
    return Dataset.from_pandas(pd.DataFrame({'input_ids': tokids, 'attention_mask': amasks, 'tokens': tokens,'tags': classes, 'labels': labels}))


In [6]:
ds = make_dataset(tokenizer, gold)
ds

Dataset({
    features: ['input_ids', 'attention_mask', 'tokens', 'tags', 'labels'],
    num_rows: 2086
})

In [7]:
def align_predictions_and_groundtruth(logits, ground_truth):
    predictions   = logits.argmax(-1)
    batch_len, *_ = predictions.shape
    overall_gnd = []
    overall_pred= []
    for sample in range(batch_len):
        sample_gnd = []
        sample_pred= []
        for gnd, pred in zip(ground_truth[sample], predictions[sample]):
            if gnd != -100:
                sample_gnd.append(id2category[gnd])
                sample_pred.append(id2category[pred])
        overall_gnd.append(sample_gnd)
        overall_pred.append(sample_pred)
    return (overall_gnd, overall_pred)


In [8]:
device     = "cuda" if torch.cuda.is_available() else "cpu"

model     = model.eval().to(device)
input_ids = torch.tensor(ds['input_ids'], dtype=torch.int).to(device)
attn_mask = torch.tensor(ds['attention_mask'], dtype=torch.int).to(device)
labels    = np.stack(ds['labels'])

N = 10
for i in range(N, len(input_ids), N):
    logits    = model(input_ids=input_ids[i-N:i], attention_mask=attn_mask[i-N:i]).logits
    print(logits.shape)

torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])
torch.Size([10, 512, 8])


KeyboardInterrupt: 